In [32]:
import numpy as np
import scipy.sparse as sp

- num_users = 3
- num_items = 5
- u1 -> i1, i2
- u2 -> i2, i4, i5
- u3 -> i3, i4

In [41]:
num_users = 3
num_items = 5

## Adjacency matrix
r_matrix = np.ndarray(shape=(num_users,num_items), dtype=np.int32)
r_matrix[0,0] = 1
r_matrix[0,1] = 1

r_matrix[1,1] = 1
r_matrix[1,3] = 1
r_matrix[1,4] = 1

r_matrix[2,2] = 1
r_matrix[2,3] = 1

r_matrix

array([[1, 1, 0, 0, 0],
       [0, 1, 0, 1, 1],
       [0, 0, 1, 1, 0]], dtype=int32)

Normalized computation using:
$$\tilde{A} = D^{-1/2} A D^{-1/2}$$
where $A$ is the adjacency matrix and D is the diagonal degree matrix, In bipartite graph (Users and Items), the adjacency matrix $A$ is structured as:
$$A = \begin{pmatrix} 0 & R \\ R^T & 0 \end{pmatrix}$$

In [42]:
def normalize_sparse_matrix(interaction_matrix):
    """
    Performs symmetric normalization for LightGCN.
    Input: interaction_matrix (R) of shape (num_users, num_items)
    """
    num_users, num_items = interaction_matrix.shape
    
    # 1. Build the Bipartite Adjacency Matrix A
    # Constructing a (U+I) x (U+I) matrix
    adj_mat = sp.dok_matrix((num_users + num_items, num_users + num_items), dtype=np.float32)
    # R is the user-item part, R.T is the item-user part
    adj_mat[:num_users, num_users:] = interaction_matrix
    adj_mat[num_users:, :num_users] = interaction_matrix.transpose()
    
    # Convert to CSR for faster arithmetic operations
    adj_mat = adj_mat.tocsr()
    
    # 2. Compute the Degree Matrix D (row sums)
    row_sum = np.array(adj_mat.sum(axis=1)).flatten()
    
    # 3. Compute D^-0.5
    # Avoid division by zero for isolated nodes
    d_inv_sqrt = np.power(row_sum, -0.5, where=row_sum != 0)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.
    
    # 4. Multiply: D^-0.5 * A * D^-0.5
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    normalized_adj = d_mat_inv_sqrt.dot(adj_mat).dot(d_mat_inv_sqrt)
    
    return normalized_adj.tocsr()

In [44]:
class LightGCNFromScratch:
    def __init__(self, interaction_matrix, embedding_dim=64, n_layers=3):
        self.num_users, self.num_items = interaction_matrix.shape
        self.emb_dim = embedding_dim
        self.n_layers = n_layers
        
        # Initialize Embeddings (User + Item) using Xavier/Normal Distribution
        # T(init) = O((U + I) * d)
        self.all_embeddings = np.random.normal(0, 0.1, (self.num_users + self.num_items, self.emb_dim))
        
        # Pre-calculate the normalized adjacency matrix
        self.norm_adj = normalize_sparse_matrix(interaction_matrix)

    def propagate(self):
        """
        The core iterative algorithm of LightGCN.
        Time Complexity: O(K * |Edges| * d)
        """
        layer_embeddings = [self.all_embeddings]
        current_embeddings = self.all_embeddings
        
        for k in range(self.n_layers):
            # Sparse Matrix-Vector Multiplication: The bottleneck step
            # O(|Edges| * d)
            current_embeddings = self.norm_adj.dot(current_embeddings)
            layer_embeddings.append(current_embeddings)
        
        # Final Readout: Simple average of all layers
        # O(K * (U + I) * d)
        final_embeddings = np.mean(layer_embeddings, axis=0)
        
        # Split back into user and item embeddings
        self.user_final_embeddings = final_embeddings[:self.num_users]
        self.item_final_embeddings = final_embeddings[self.num_users:]
        
        return self.user_final_embeddings, self.item_final_embeddings

    def recommend(self, user_id, top_k=10):
        """
        Generates recommendations using Dot Product similarity.
        """
        user_emb = self.user_final_embeddings[user_id]
        # Calculate scores for all items: O(Items * d)
        scores = np.dot(self.item_final_embeddings, user_emb)
        
        # Use a partition/heap to find top K: O(Items * log K)
        top_indices = np.argpartition(scores, -top_k)[-top_k:]
        return top_indices[np.argsort(scores[top_indices])[::-1]]

In [45]:
model_lgcn = LightGCNFromScratch(r_matrix)

In [50]:
print(f"n_users:{model_lgcn.num_users},n_items: {model_lgcn.num_items}, emb:{model_lgcn.emb_dim}, layers:{model_lgcn.n_layers}")

print(f"all_embeddings:{model_lgcn.all_embeddings.shape}")

n_users:3,n_items: 5, emb:64, layers:3
all_embeddings:(8, 64)


In [51]:
u_final_emb, i_final_emb = model_lgcn.propagate()

In [56]:
rec_items = model_lgcn.recommend(user_id=0,top_k=3)

- num_users = 3
- num_items = 5
- u0 -> i0, i1
- u1 -> i1, i3, i4
- u2 -> i2, i3

In [57]:
rec_items

array([0, 1, 3])

In [ ]:
import numpy as np

class GraphSAGEFromScratch:
    def __init__(self, adj_list, features, sample_sizes=[10, 5], emb_dim=64):
        """
        adj_list: Dictionary {node: [neighbors]}
        features: Initial feature matrix (U+I, d)
        sample_sizes: Number of neighbors to sample per layer
        """
        self.adj_list = adj_list
        self.features = features
        self.sample_sizes = sample_sizes
        self.emb_dim = emb_dim
        # Learnable weights for each layer
        self.weights = [np.random.normal(0, 0.1, (features.shape[1] * 2, emb_dim)) 
                        for _ in sample_sizes]

    def sample_neighbors(self, nodes, size):
        """Step 1: Sampling Algorithm (O(Nodes * Size))"""
        sampled_neighbors = []
        for node in nodes:
            neighbors = self.adj_list.get(node, [])
            if len(neighbors) >= size:
                sampled = np.random.choice(neighbors, size, replace=False)
            else:
                # Handle case where node has fewer neighbors than sample size
                sampled = np.random.choice(neighbors, size, replace=True) if neighbors else [node]*size
            sampled_neighbors.append(sampled)
        return np.array(sampled_neighbors)

    def aggregate_mean(self, neighbor_embs):
        """Step 2: Aggregator Algorithm (O(Nodes * Size * d))"""
        return np.mean(neighbor_embs, axis=1)

    def forward(self, nodes):
        """
        The forward pass for a batch of nodes. 
        Demonstrates the recursive nature of SAGE.
        """
        curr_nodes = np.array(nodes)
        node_features = self.features[curr_nodes]

        for layer, size in enumerate(self.sample_sizes):
            # 1. Sample neighbors for current nodes
            neighbor_indices = self.sample_neighbors(curr_nodes, size)
            
            # 2. Get features of neighbors
            neighbor_feats = self.features[neighbor_indices.flatten()].reshape(len(curr_nodes), size, -1)
            
            # 3. Aggregate neighbor info
            aggregated = self.aggregate_mean(neighbor_feats)
            
            # 4. Concatenate self and neighbor info (Combine Step)
            combined = np.hstack([node_features, aggregated])
            
            # 5. Transform and Non-linear activation (O(d^2))
            node_features = np.tanh(combined @ self.weights[layer])
            
        return node_features

In [ ]:
def recommend_sage(model, user_id, all_item_ids, top_k=10):
    """
    model: Your GraphSAGEFromScratch instance
    user_id: The ID of the user we are recommending for
    all_item_ids: List of all possible item IDs in the dataset
    """
    # 1. Generate User Embedding on-the-fly (Inductive step)
    # We pass the user_id through the SAGE forward pass (sampling neighbors)
    user_emb = model.forward([user_id]) # Shape: (1, emb_dim)
    
    # 2. Generate Item Embeddings
    # In a real system, item embeddings are often pre-computed 
    # but let's assume we compute them here
    item_embs = model.forward(all_item_ids) # Shape: (num_items, emb_dim)
    
    # 3. Compute Scores (Dot Product)
    # T(score) = O(M * d)
    scores = np.dot(item_embs, user_emb.T).flatten()
    
    # 4. Rank using a Max-Heap approach (represented by argpartition)
    # T(rank) = O(M log K)
    top_indices = np.argpartition(scores, -top_k)[-top_k:]
    sorted_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
    
    return all_item_ids[sorted_indices], scores[sorted_indices]

In [ ]:
class GATFromScratch:
    def __init__(self, adj_matrix, features, n_heads=1):
        self.adj = adj_matrix # Sparse matrix (U+I, U+I)
        self.features = features
        self.n_heads = n_heads
        input_dim = features.shape[1]
        
        # Learnable parameters
        self.W = np.random.normal(0, 0.1, (input_dim, input_dim))
        self.a = np.random.normal(0, 0.1, (2 * input_dim, 1))

    def leaky_relu(self, x, alpha=0.2):
        return np.where(x > 0, x, alpha * x)

    def compute_attention(self, h):
        """
        Step 1 & 2: Calculate attention coefficients for all edges.
        Complexity: O(|Edges| * d)
        """
        num_nodes = h.shape[0]
        # For each edge (i, j), compute e_ij
        rows, cols = self.adj.nonzero()
        
        # Concatenate features of source and target nodes
        # a.T * [Wh_i || Wh_j]
        combined = np.hstack([h[rows], h[cols]])
        e = self.leaky_relu(combined @ self.a)
        
        # Step 3: Softmax over neighbors
        # To do this from scratch, we build a sparse attention matrix
        exp_e = np.exp(e)
        sum_exp = np.array(self.adj.copy().astype(float).tocsr()) # Template
        # ... logic to normalize exp_e per row ...
        
        return exp_e # Simplified for pseudocode

    def forward(self):
        # Linear transformation
        h = self.features @ self.W
        
        # Attention weights
        alpha = self.compute_attention(h)
        
        # Step 4: Weighted Aggregation
        # New features are sum(alpha_ij * h_j)
        # T(agg) = O(|Edges| * d)
        h_prime = self.adj.multiply(alpha).dot(h)
        
        return np.tanh(h_prime)

In [ ]:
def recommend_gat(model, user_id, top_k=10):
    """
    model: Your GATFromScratch instance
    user_id: The ID of the user
    """
    # 1. Run the full GAT forward pass to get updated embeddings
    # This includes the O(|E| * d) attention calculations
    all_updated_features = model.forward()
    
    # 2. Extract User and Item vectors from the combined matrix
    user_emb = all_updated_features[user_id]
    # Assuming item embeddings start after user indices
    item_embs = all_updated_features[model.num_users:] 
    
    # 3. Calculate Alignment/Similarity
    scores = np.dot(item_embs, user_emb)
    
    # 4. Get Top-K
    top_item_ids = np.argpartition(scores, -top_k)[-top_k:]
    return top_item_ids[np.argsort(scores[top_item_ids])[::-1]]